# NDSharp Generic Math — Usage and Reference
Storage in `T`, views and MathemaSharp kernels without `double` intermediates. This notebook uses the local Release assemblies, not an already published package.
`NDArray<T>` sits alongside the runtime container: it does not replace its dtype and NumPy API.

In [ ]:
#load "../shared/MathemaSharp.csx"
#load "../shared/NDSharp.csx"
using System;
using System.Linq;
using System.Numerics;
using NDSharp;
using MathemaSharp.NDSharp;
void Require(bool condition, string label) { if (!condition) throw new Exception(label); Console.WriteLine(label + ": OK"); }

## Usage: storage, slicing and broadcasting
The constructor copies the data and the shape. Slice and transposition share writes. `Slice(axis,start,length,step)` uses an explicit length, not an end index. `Add` creates a new array and aligns the trailing axes.

In [ ]:
var delta = Decimal128.Parse("1e-25");
var typed = new Decimal128[] { Decimal128.One, delta }.ToTypedNDArray();
Require(typed.Sum() - Decimal128.One == delta, "decimal128_storage");
var vector = new NDArray<int>(new int[] {1,2,3}, 3);
var reverse = vector.Slice(0,2,3,-1);
Require(reverse.Add(new NDArray<int>(new int[] {10})).ToArray().SequenceEqual(new int[] {13,12,11}), "broadcast_reverse");
reverse[0] = 9;
Require(vector[2] == 9, "shared_view_write");

decimal128_storage: OK
broadcast_reverse: OK
shared_view_write: OK

## Usage: products and linear systems
`Dot` is rank 1 and non-conjugating; `MatMul` is rank 2, without batching. Ordinary accumulation in `T`, not compensated. The bridge copies the views before calling the existing LU, QR and diagnostics.

In [ ]:
var a = new double[,] {{2,0},{0,4}}.ToTypedNDArray();
var b = new double[] {2,8}.ToTypedNDArray();
var lu = a.FactorizeLu();
Require(lu.Solve(b).ToArray().SequenceEqual(new double[] {1,2}), "lu_solution");
Require(a.MatMul(a.Transpose()).ToArray().SequenceEqual(new double[] {4,0,0,16}), "matrix_product");
var report = a.SolveWithDiagnostics(b);
Require(report.ConditionInfinityEstimate == 2 && !report.IsIllConditioned, "diagnostics");
var q = a.LeastSquares(b).ToArray();
Require(Math.Abs(q[0]-1) < 1e-12 && Math.Abs(q[1]-2) < 1e-12, "qr_solution");

lu_solution: OK
matrix_product: OK
diagnostics: OK
qr_solution: OK

In [ ]:
void Check<T>() where T : IFloatingPointIeee754<T> {
 var v = new T[] {T.One,T.CreateChecked(2),T.CreateChecked(3)}.ToTypedNDArray();
 Require(v.Dot(v) == T.CreateChecked(14), typeof(T).Name + "_dot");
 var m = new T[,] {{T.One,T.Zero},{T.Zero,T.One}}.ToTypedNDArray();
 var rhs = new T[] {T.One,T.CreateChecked(2)}.ToTypedNDArray();
 Require(m.FactorizeLu().Solve(rhs).ToArray().SequenceEqual(rhs.ToArray()), typeof(T).Name + "_lu");
}
Check<Half>(); Check<float>(); Check<double>(); Check<Decimal32>(); Check<Decimal64>(); Check<Decimal128>();

Half_dot: OK
Half_lu: OK
Single_dot: OK
Single_lu: OK
Double_dot: OK
Double_lu: OK
Decimal32_dot: OK
Decimal32_lu: OK
Decimal64_dot: OK
Decimal64_lu: OK
Decimal128_dot: OK
Decimal128_lu: OK

## Reference
| API | Contract |
|---|---|
| `NDArray<T>` | `INumber<T>`; rank zero = scalar, empty dimensions allowed |
| `Shape`, `Strides`, `ToArray` | Independent copies; ToArray respects logical order |
| `Slice`, `Transpose` | Shared views; indices and permutations validated |
| `Sum`, `Add`, `Dot`, `MatMul` | Arithmetic and overflow of type T; no universal accuracy |
| `FactorizeLu`, `LeastSquares`, `SolveWithDiagnostics` | `IFloatingPointIeee754<T>`; contracts of the LinearAlgebra kernels |

Conditioning is a warning, not a certification of the error. FFT, wavelets, distributions, automatic runtime/generic conversion and batched products are excluded.
Kernel reference: [LinearAlgebra](../LinearAlgebra/12-api-reference.ipynb).